---
title: "Forecast Sale"
jupyter : python3 
---

# Import Modules

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df= pd.read_csv("Dataset/MonthlySaleBakery.csv")
df['year'] =df['year'].astype(str)
df.dtypes

### Creating a new coloumns

In [ ]:
df['date'] = df['month']+ '-' + df['year']

### Ploting the sale times series

In [ ]:
fig , ax = plt.subplots(figsize = (15 ,5))
ax.plot(df['date'] , df['Monthly Sale'] , marker ='o' , linestyle ='--' , linewidth = 2 , markersize = 6 , color ='#1f77b4')
ax.tick_params(bottom = False , left = False)
ax.tick_params(axis ='x' , rotation = 45)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _ : f"{x/1e3} K"))
ax.spines[['top', 'right']].set_visible(False)
ax.spines[['bottom', 'left']].set_color('#898989')
ax.grid(axis='y', color = '#898989' , linewidth = 0.8)
ax.set_axisbelow(True)
ax.set_xlabel("Month" , labelpad = 10  , fontsize = 12 )
ax.set_ylabel("Sales ($)" , labelpad = 10 , fontsize = 12 )
ax.set_title("Top-selling Products, Jan 2021 to Sep 2022", fontsize = 14 , pad = 15)

### Import Sklearn Modules

In [ ]:
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_error , mean_absolute_percentage_error  

In [ ]:
ts = df.set_index('date')['Monthly Sale']
ts.index.freq = 'MS'

## Prepare the Data 

In [ ]:
h = 3 
train , test =ts[:-h] , ts[-h:]
print(test.index)

## Forecast Model 

### Seasonal naive with growth adjustment

In [ ]:
growth = train['Jan-2022':'Jun-2022'].sum()/train['Jan-2021':'Jun-2021'].sum()
snaive_pred = ts.shift(12)[test.index]* growth

In [ ]:
print(growth)

In [ ]:
m = pd.to_datetime(df['date'], format='%b-%Y').dt.month.values
x = pd.DataFrame({'trend':np.arange(len(df['date'].values))} , index= df['date'])
K = 3 
for k in range(1 , K+1):
  x[f'sin{k}'] = np.sin(2 * np.pi * k * m /12)
  x[f'cos{k}'] = np.cos(2 * np.pi * k * m /12)
x.insert(0 , 'const' , 1.0)

fit = sm.OLS(train , x.loc[train.index]).fit()
fourier_pred = fit.predict(x.loc[test.index])

### Print the output

In [ ]:
print(fourier_pred)

### Compare throught MAE

In [ ]:
results = pd.Series({
  'Seasonal Naive + Growth':                mean_absolute_error(test , snaive_pred),
  'Fourier Regression':                     mean_absolute_error(test , fourier_pred),

} , name = 'MAE').sort_values()
print(results)

### Compare throught MAPE 

In [ ]:
results = pd.Series({
  'Seasonal Naive + Growth':                mean_absolute_percentage_error(test , snaive_pred),
  'Fourier Regression':                     mean_absolute_percentage_error(test , fourier_pred),

} , name = 'MAPE').sort_values()
print(results)

## Discussion 

As you can see from the result **Seasonal Naive + Growth** show a lower MSE than Fourier Regression model. So I will be implementing Seasonal Naive + Growth model 

In [ ]:
growth = ts['Jan-2022':'Sep-2022'].sum() / ts['Jan-2021':'Sep-2021'].sum()
forecast = ts['Oct-2021':'Dec-2021'] * growth
forecast.index = pd.date_range('2022-10-01', periods=3, freq='MS')
forecast.index = forecast.index.strftime('%b-%Y')
print(forecast)

### Visualize the data

In [ ]:
fig , ax = plt.subplots(figsize = (15 ,5))
ax.plot(df['date'] , df['Monthly Sale'] , marker ='o' , linestyle ='--' ,label='Actual')

## Forecast Data
last_data = df['date'].iloc[-1]
last_value = df['Monthly Sale'].iloc[-1]

fc_ind = [last_data] + list(forecast.index)
fc_val = [last_value] + list(forecast.values)

ax.plot(fc_ind , fc_val , marker ='o' , linestyle ='--' , color='orange' , label ='Forecast')

ax.tick_params(bottom = False , left = False)
ax.tick_params(axis ='x' , rotation = 45)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _ : f"{x/1e3} K"))
ax.spines[['top', 'right']].set_visible(False)
ax.spines[['bottom', 'left']].set_color('#898989')
ax.grid(True , color = '#898989')
ax.set_xlabel("Time Horizon" , labelpad = 25 , fontsize = 12 )
ax.set_ylabel("Sale ($)" , labelpad = 25 , fontsize = 12 )
ax.set_title("Top-Selling Products: Actual and Forecast (Jan 2021 to Dec 2022)")